# 04. Feature Engineering V2 — 빈도 기반 선택 성분

**목적**  
8개 이상 제품에 등장하고 전체 제품의 80% 이하에서 등장한 성분을 선택합니다.

**입력**  
`data/interim/전성분_표준화_최종.csv`

**출력**  
`data/interim/V2_선택성분_원핫인코딩.csv 및 선택 성분 목록`

> 저장된 전처리 데이터만 사용하며 외부 요청은 발생하지 않습니다.


In [ ]:
from pathlib import Path

# Jupyter와 Colab 모두 저장소 루트에서 실행합니다.
def find_project_root(start=Path.cwd()):
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "data").is_dir() and (path / "notebooks").is_dir():
            return path
    raise FileNotFoundError("저장소를 clone한 뒤 해당 폴더 안에서 실행하세요.")

PROJECT_ROOT = find_project_root()

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

for directory in [DATA_RAW_DIR, DATA_INTERIM_DIR, DATA_PROCESSED_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)


# KDMS V2 — 실제 성분명 기반 One-hot X 데이터셋

V2는 숫자형 피처 ID가 아니라, 필터를 통과한 **실제 성분명 자체를 컬럼명**으로 사용합니다.

## 처리 기준

1. 동일 제품 내 중복 성분 제거
2. 전체 제품 중 **8개 이상**에서 등장한 성분만 유지
3. 전체 제품의 **80% 초과**에서 등장하는 지나치게 흔한 주성분 제거
4. 남은 성분을 실제 성분명으로 One-hot

예시:

```text
v2_나이아신아마이드
v2_세라마이드엔피
v2_판테놀
v2_에틸헥실글리세린
```

제품에 해당 성분이 있으면 `1`, 없으면 `0`입니다.

현재 231개 제품 기준 예상:

- 전체 고유 성분: 1,292개
- 8개 이상 등장: 252개
- 80% 초과 주성분 제거: 4개
- 최종 V2 성분: 248개
- 제품×피부유형: 1,617행


In [ ]:
from pathlib import Path
import hashlib
import re
import unicodedata

import pandas as pd

LONG_PATH = DATA_INTERIM_DIR / "전성분_표준화_최종.csv"
OUTPUT_DIR = DATA_INTERIM_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_OUTPUT_PATH = OUTPUT_DIR / "V2_선택성분_원핫인코딩.csv"
SELECTED_OUTPUT_PATH = OUTPUT_DIR / "V2_선택성분_목록.csv"

MIN_PRODUCT_COUNT = 8
MAX_PRODUCT_RATIO = 0.80

SKIN_TYPES = [
    "건성", "약건성", "지성", "복합성", "민감성", "트러블성", "중성"
]

print("전성분:", LONG_PATH)
print("출력:", OUTPUT_DIR)


## 1. 전성분 정리


In [ ]:
long_df = pd.read_csv(
    LONG_PATH,
    encoding='utf-8-sig',
)

required_columns = {
    'product_id',
    'product_name',
    'canonical_name',
}

missing_columns = (
    required_columns - set(long_df.columns)
)

if missing_columns:
    raise ValueError(
        f'필수 컬럼 누락: {sorted(missing_columns)}'
    )

if 'product_group_id' in long_df.columns:
    ID_COL = 'product_group_id'
else:
    ID_COL = 'product_id'

long_df[ID_COL] = (
    long_df[ID_COL]
    .astype(str)
    .str.strip()
)

long_df['canonical_name'] = (
    long_df['canonical_name']
    .astype('string')
    .map(
        lambda value: (
            unicodedata.normalize(
                'NFKC',
                str(value),
            ).strip()
            if pd.notna(value)
            else value
        )
    )
)


In [ ]:
ingredient_base = (
    long_df[
        [
            ID_COL,
            'product_name',
            'canonical_name',
        ]
    ]
    .dropna(
        subset=[ID_COL, 'canonical_name']
    )
    .loc[
        lambda frame:
        frame['canonical_name'].ne('')
    ]
    .drop_duplicates(
        [ID_COL, 'canonical_name']
    )
    .reset_index(drop=True)
)

product_info = (
    ingredient_base[
        [ID_COL, 'product_name']
    ]
    .drop_duplicates(ID_COL)
    .sort_values(ID_COL)
    .reset_index(drop=True)
)

n_products = product_info[ID_COL].nunique()

print('사용 ID:', ID_COL)
print('제품 수:', n_products)
print(
    '전체 고유 성분:',
    ingredient_base['canonical_name'].nunique(),
)


사용 ID: product_id
제품 수: 231
전체 고유 성분: 1292


## 2. 8개 이상·80% 이하 성분 선정


In [ ]:
frequency_df = (
    ingredient_base
    .groupby('canonical_name')[ID_COL]
    .nunique()
    .rename('product_count')
    .reset_index()
)

frequency_df['product_ratio'] = (
    frequency_df['product_count']
    / n_products
)

selected_df = (
    frequency_df[
        (
            frequency_df['product_count']
            >= MIN_PRODUCT_COUNT
        )
        & (
            frequency_df['product_ratio']
            <= MAX_PRODUCT_RATIO
        )
    ]
    .sort_values(
        ['product_count', 'canonical_name'],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

def make_feature_name(
    ingredient_name,
):
    text = unicodedata.normalize(
        'NFKC',
        str(ingredient_name),
    ).strip()

    text = re.sub(
        r'[\r\n\t]+',
        ' ',
        text,
    )
    text = re.sub(
        r'\s+',
        ' ',
        text,
    )

    return f'v2_{text}'

selected_df['feature_name'] = (
    selected_df['canonical_name']
    .map(make_feature_name)
)


In [ ]:
if not selected_df['feature_name'].is_unique:
    duplicated = selected_df.loc[
        selected_df['feature_name']
        .duplicated(keep=False),
        [
            'canonical_name',
            'feature_name',
        ],
    ]

    raise ValueError(
        '성분명 기반 컬럼명이 중복됩니다.\n'
        f'{duplicated}'
    )

print('전체 고유 성분:', len(frequency_df))
print(
    '8개 이상 등장:',
    int(
        (
            frequency_df['product_count']
            >= MIN_PRODUCT_COUNT
        ).sum()
    ),
)
print(
    '80% 초과로 제거:',
    int(
        (
            (
                frequency_df['product_count']
                >= MIN_PRODUCT_COUNT
            )
            & (
                frequency_df['product_ratio']
                > MAX_PRODUCT_RATIO
            )
        ).sum()
    ),
)
print('최종 V2 성분:', len(selected_df))

display(
    selected_df[
        [
            'feature_name',
            'canonical_name',
            'product_count',
            'product_ratio',
        ]
    ].head(30)
)


전체 고유 성분: 1292
8개 이상 등장: 252
80% 초과로 제거: 4
최종 V2 성분: 248


,feature_name,canonical_name,product_count,product_ratio
0,v2_에틸헥실글리세린,에틸헥실글리세린,162,0.701299
1,v2_판테놀,판테놀,138,0.597403
2,v2_카프릴릭/카프릭트라이글리세라이드,카프릴릭/카프릭트라이글리세라이드,137,0.593074
3,v2_토코페롤,토코페롤,135,0.584416
4,v2_소듐하이알루로네이트,소듐하이알루로네이트,132,0.571429
5,v2_나이아신아마이드,나이아신아마이드,119,0.515152
6,v2_프로판다이올,프로판다이올,119,0.515152
7,v2_하이드로제네이티드레시틴,하이드로제네이티드레시틴,117,0.506494
8,v2_아데노신,아데노신,115,0.497835
9,v2_세테아릴알코올,세테아릴알코올,111,0.480519


## 3. 실제 성분명 컬럼으로 One-hot


In [ ]:
ingredient_to_feature = dict(
    zip(
        selected_df['canonical_name'],
        selected_df['feature_name'],
    )
)

selected_long = ingredient_base[
    ingredient_base['canonical_name']
    .isin(selected_df['canonical_name'])
].copy()

selected_long['feature_name'] = (
    selected_long['canonical_name']
    .map(ingredient_to_feature)
)

onehot_df = (
    selected_long
    .assign(value=1)
    .pivot_table(
        index=ID_COL,
        columns='feature_name',
        values='value',
        aggfunc='max',
        fill_value=0,
    )
    .reindex(
        index=product_info[ID_COL],
        columns=selected_df['feature_name'],
        fill_value=0,
    )
    .astype('int8')
    .reset_index()
)

feature_columns = (
    selected_df['feature_name']
    .tolist()
)

product_x = (
    product_info
    .merge(
        onehot_df,
        on=ID_COL,
        how='left',
        validate='one_to_one',
    )
)

product_x[feature_columns] = (
    product_x[feature_columns]
    .fillna(0)
    .astype('int8')
)

print('제품 단위 One-hot:', product_x.shape)


In [ ]:
print(
    '실제 V2 피처 수:',
    len(feature_columns),
)


제품 단위 One-hot: (231, 250)
실제 V2 피처 수: 248


## 4. 동일 처방 Group과 피부유형 확장


In [ ]:

product_ingredient_sets = (
    ingredient_base
    .groupby(ID_COL)['canonical_name']
    .apply(set)
)

def formula_hash(ingredient_set):
    formula_text = '\x1f'.join(
        sorted(ingredient_set)
    )

    return hashlib.md5(
        formula_text.encode('utf-8')
    ).hexdigest()

formula_group_df = (
    product_ingredient_sets
    .map(formula_hash)
    .rename('formula_group')
    .reset_index()
)

product_x = product_x.merge(
    formula_group_df,
    on=ID_COL,
    how='left',
    validate='one_to_one',
)

skin_df = pd.DataFrame({
    '피부타입': SKIN_TYPES,
})

product_x['_join_key'] = 1
skin_df['_join_key'] = 1

v2_x = (
    product_x
    .merge(
        skin_df,
        on='_join_key',
        how='inner',
    )
    .drop(columns='_join_key')
)

v2_x = v2_x[
    [
        ID_COL,
        'product_name',
        '피부타입',
        'formula_group',
    ]
    + feature_columns
]

print('최종 V2 X 크기:', v2_x.shape)
display(v2_x.head())


최종 V2 X 크기: (1617, 252)


,product_id,product_name,피부타입,formula_group,v2_에틸헥실글리세린,v2_판테놀,v2_카프릴릭/카프릭트라이글리세라이드,v2_토코페롤,v2_소듐하이알루로네이트,v2_나이아신아마이드,...,v2_옥수수전분,v2_카놀라오일,v2_카카오씨추출물,v2_캐모마일꽃오일,v2_커피콩추출물,v2_트레오닌,v2_티트리잎추출물,v2_팔미토일테트라펩타이드-7,v2_폴리글리세릴-2스테아레이트,v2_폴리아이소부텐
0,A000000002848,바이오더마 세비엄 포어 리파이너,건성,f5f6b9ab23517e277ec36cdb5c3aeb67,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,A000000002848,바이오더마 세비엄 포어 리파이너,약건성,f5f6b9ab23517e277ec36cdb5c3aeb67,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,A000000002848,바이오더마 세비엄 포어 리파이너,지성,f5f6b9ab23517e277ec36cdb5c3aeb67,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,A000000002848,바이오더마 세비엄 포어 리파이너,복합성,f5f6b9ab23517e277ec36cdb5c3aeb67,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,A000000002848,바이오더마 세비엄 포어 리파이너,민감성,f5f6b9ab23517e277ec36cdb5c3aeb67,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 5. 저장 및 QA


In [ ]:
selected_output = selected_df[
    [
        'feature_name',
        'canonical_name',
        'product_count',
        'product_ratio',
    ]
].copy()

selected_output['min_product_count'] = (
    MIN_PRODUCT_COUNT
)
selected_output['max_product_ratio'] = (
    MAX_PRODUCT_RATIO
)
selected_output['n_products'] = (
    n_products
)

v2_x.to_csv(
    FEATURE_OUTPUT_PATH,
    index=False,
    encoding='utf-8-sig',
)

selected_output.to_csv(
    SELECTED_OUTPUT_PATH,
    index=False,
    encoding='utf-8-sig',
)

expected_rows = (
    n_products * len(SKIN_TYPES)
)

assert len(v2_x) == expected_rows
assert len(feature_columns) == len(
    selected_output
)
assert v2_x[
    [ID_COL, '피부타입']
].duplicated().sum() == 0
assert v2_x[
    feature_columns
].isna().sum().sum() == 0

unique_values = set(
    pd.unique(
        v2_x[
            feature_columns
        ].to_numpy().ravel()
    )
)

assert unique_values.issubset({0, 1})

print('저장 완료')
print('1.', FEATURE_OUTPUT_PATH)
print('2.', SELECTED_OUTPUT_PATH)


In [ ]:
print('\n최종 QA')
print('- 제품 수:', n_products)
print('- 피부유형 수:', len(SKIN_TYPES))
print('- 총 행 수:', len(v2_x))
print('- 실제 성분명 피처 수:', len(feature_columns))
print('- 결측값:', v2_x.isna().sum().sum())
print('- One-hot 값:', sorted(unique_values))


## 6. Master X 결합

V4와 다음 Key로 결합합니다.

```python
master_x = v4_x.merge(
    v2_x[
        ['product_id', '피부타입']
        + [
            column
            for column in v2_x.columns
            if column.startswith('v2_')
        ]
    ],
    on=['product_id', '피부타입'],
    how='inner',
    validate='one_to_one',
)
```

V2 조합 선택:

```python
v2_columns = [
    column
    for column in master_x.columns
    if column.startswith('v2_')
]
```

각 V2 컬럼명이 실제 성분명이므로 별도의 숫자 ID Dictionary 없이도
모델 중요도와 결과를 바로 해석할 수 있습니다.
